In [1]:
import jax
import jax.numpy as jnp
import jax.random as jrandom

import time
import os
from IPython.display import display

jax.default_backend()
key = jrandom.key(1)

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

In [2]:
N = 1_00_000
x = jnp.arange(0, N).astype(jnp.float32)
display(x[:10], x[-5:-1])

Array([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.], dtype=float32)

Array([99995., 99996., 99997., 99998.], dtype=float32)

In [3]:
# Fibonacci Number

# Python Implementation
def fibonacci_python():
    fb1, fb2 = 0, 1 
    f = [0]*N
    for i in range(2, N):
        fib = fb1 + fb2
        fb1 = fb2
        fb2 = fib
        f[i] = fib

# Jax Internal Implementation
def fibonacci_jax_internal(carry, _):
    fi1, fi2 = carry
    return (fi2, fi1+fi2), None

# UnJitted Jax Function
def fibonacci_jax():
    final, _ = jax.lax.scan(fibonacci_jax_internal, (0, 1), None, length=N)
    return final

# Jitted Jax Function
fibonacci_jax_jitted = jax.jit(fibonacci_jax)

In [4]:
def benchmark_fibonacci_python():
    # Begin Timer
    begin = time.perf_counter()
    # Invoke the Actual Function
    fibonacci_python()
    # End Timer
    end = time.perf_counter()
    # Metrics
    display(f"Elapsed Time = {end - begin}")


def benchmark_fibonacci_jax_jitted():
    # Initial Warmup
    result_warmup = fibonacci_jax_jitted()
    result_warmup[0].block_until_ready()

    # Begin Timer
    begin = time.perf_counter()
    # Invoke the Actual Function
    result_timed = fibonacci_jax_jitted()
    # Wait Until Ready
    result_timed[0].block_until_ready()
    # End Timer
    end = time.perf_counter()
    # Metric
    display(f"Elapsed Time = {end - begin}")

benchmark_fibonacci_python()
benchmark_fibonacci_jax_jitted()

'Elapsed Time = 0.24282997999398503'

'Elapsed Time = 0.6842349549988285'

In [5]:
# Newton Method

# Some function f(x)
def f(x: jax.Array):
    return jnp.pow(x, 2) - 8

# Derivative of that function
fdash = jax.grad(f)

# Jax Internal Function
def newton_method_jax_internal(current_guess: jax.Array, _):
    new_guess = current_guess - f(current_guess) / fdash(current_guess)
    return new_guess, new_guess

# Number of Iterations
NITERATIONS = 1_00_000

In [6]:
# Python Implementation
def newton_method_python():
    guess = 1.0
    for i in range(NITERATIONS):
        guess = guess - f(guess)/ fdash(guess)
    return guess

# Jax Unjitted Implementation
def newton_method_jax():
    guess = 1.0
    root, _ = jax.lax.scan(newton_method_jax_internal, guess, None, length=NITERATIONS)
    return root
# Jax Jitted Implementation
newton_method_jax_jitted = jax.jit(newton_method_jax)

In [8]:
def benchmark_newton_method_python():
    # Begin Timer
    begin = time.perf_counter()
    # Invoke the Actual Function
    root = newton_method_python()
    # End Timer
    end = time.perf_counter() 
    # Metrics
    display(f"Elapsed Time = {end - begin}")
    display(f"Final root = {root}")

def benchmark_newton_method_jax():
     # Begin Timer
    begin = time.perf_counter()
    # Invoke the Actual Function
    root = newton_method_jax()
    # End Timer
    end = time.perf_counter() 
    # Metrics
    display(f"Elapsed Time = {end - begin}")
    display(f"Final root = {root}")

def benchmark_newton_method_jax_jitted():
     # Begin Timer
    begin = time.perf_counter()
    # Invoke the Actual Function
    root = newton_method_jax_jitted()
    # End Timer
    end = time.perf_counter() 
    # Metrics
    display(f"Elapsed Time = {end - begin}")
    display(f"Final root = {root}")

benchmark_newton_method_python()
benchmark_newton_method_jax()
benchmark_newton_method_jax_jitted()

'Elapsed Time = 177.67596383600176'

'Final root = 2.8284270763397217'

'Elapsed Time = 0.7455800759998965'

'Final root = 2.8284270763397217'

'Elapsed Time = 0.31405987999460194'

'Final root = 2.8284270763397217'